# Notebook 05 — LLM avanzati: ragionamento, multimodalità, ricerca

**Modulo:** 05
**Tempo stimato:** 2-3 ore

## Cosa costruirai

Al termine di questo notebook avrai:
- Confrontato con i tuoi occhi una risposta "diretta" e una con ragionamento esteso attivato
- Analizzato un documento lungo caricato come contesto (multimodalità testuale)
- Costruito un piccolo assistente con ricerca web integrata (grounding)

## Prerequisiti

```bash
pip install anthropic python-dotenv
```

Serve un file `.env` nella cartella del progetto con:
```
ANTHROPIC_API_KEY=sk-ant-...
```


---
## Cella 0 — Setup: caricamento delle chiavi

Esegui sempre questa cella per prima.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

ANTHROPIC_KEY = os.getenv("ANTHROPIC_API_KEY")

def verifica_chiave(nome: str, chiave: str | None) -> None:
    if chiave:
        print(f"OK {nome}: {chiave[:8]}...")
    else:
        print(f"MANCANTE {nome}: controlla il tuo file .env")

verifica_chiave("ANTHROPIC_API_KEY", ANTHROPIC_KEY)


---
## Cella 1 — Domanda complessa in modalità standard

Poniamo a Claude un problema che richiede qualche passaggio logico, **senza**
attivare il ragionamento esteso. Osserva la risposta: potrebbe essere corretta,
ma il modello non mostra alcun passaggio intermedio.

**Prima di eseguire:** secondo te, un problema con più vincoli da bilanciare
verrà risolto correttamente al primo colpo, in modalità standard?

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)

PROBLEMA = (
    "Un magazzino riceve 3 pallet da 48 scatole ciascuno. Il 15% delle scatole "
    "risulta danneggiato e viene scartato. Le scatole rimaste vengono divise in "
    "parti uguali su 9 camion. Quante scatole intere carica ogni camion, e quante "
    "scatole restano a magazzino perché non si dividono in parti uguali? "
    "Mostra solo il risultato finale, senza spiegazioni."
)

risposta_diretta = client.messages.create(
    model="claude-sonnet-5",
    max_tokens=512,
    messages=[{"role": "user", "content": PROBLEMA}]
)

testo_diretto = risposta_diretta.content[0].text
print("Risposta (modalità standard):")
print(testo_diretto)


---
## Cella 2 — La stessa domanda con ragionamento esteso attivato

Attiviamo il pensiero adattivo (`thinking={"type": "adaptive"}`) e chiediamo
anche un riassunto del ragionamento (`display: "summarized"`) per poterlo
osservare. Confronta il risultato numerico con quello della cella precedente.

In [ ]:
risposta_ragionata = client.messages.create(
    model="claude-sonnet-5",
    max_tokens=2048,
    thinking={"type": "adaptive", "display": "summarized"},
    messages=[{"role": "user", "content": PROBLEMA}]
)

for blocco in risposta_ragionata.content:
    if blocco.type == "thinking":
        print("--- Ragionamento (riassunto) ---")
        print(blocco.thinking[:800], "...\n")
    elif blocco.type == "text":
        print("--- Risposta finale ---")
        print(blocco.text)

print(f"\nToken di output totali (pensiero + risposta): {risposta_ragionata.usage.output_tokens}")


**Le tue osservazioni** (modifica questa cella):

- Il risultato numerico delle due celle precedenti coincide? ...
- Quanti token in più ha consumato la versione con ragionamento esteso? ...
- Per questo tipo di problema (aritmetica a più passaggi), il ragionamento
  esteso ti sembra necessario o è uno spreco? ...
- In quali categorie di compiti, secondo te, il ragionamento esteso fa la
  differenza più grande? ...

---
## Cella 3 — Analisi di un documento lungo come contesto

Simuliamo un caso reale: hai un documento lungo (un estratto di policy
aziendale) e vuoi che il modello ne estragga informazioni strutturate, senza
doverlo spezzare a mano. Questo è lo stesso principio dell'analisi di un PDF
di 40 pagine — qui usiamo un testo incorporato per restare autosufficienti,
ma il procedimento con un PDF caricato è concettualmente identico (vedi
Files API, che approfondiremo più avanti nel corso).

In [ ]:
DOCUMENTO = """
POLICY INTERNA — GESTIONE DELLE FERIE E DEI PERMESSI (estratto)

Articolo 1 — Maturazione delle ferie
Ogni dipendente matura 26 giorni di ferie annuali, calcolati pro-rata per chi
è assunto durante l'anno. Le ferie maturate nell'anno N devono essere godute
entro il 30 giugno dell'anno N+1, salvo accordi specifici con il proprio
responsabile.

Articolo 2 — Richiesta delle ferie
Le richieste di ferie superiori a 3 giorni consecutivi vanno inoltrate con
almeno 15 giorni di preavviso tramite il portale HR. Richieste per periodi
inferiori a 3 giorni possono essere inoltrate con 48 ore di preavviso.

Articolo 3 — Permessi retribuiti
Sono previsti permessi retribuiti per: visite mediche documentate (fino a 4
ore per evento), lutto di parente entro il secondo grado (3 giorni), e
donazione di sangue (l'intera giornata). I permessi vanno registrati entro 5
giorni lavorativi dall'evento.

Articolo 4 — Smart working
Ogni dipendente ha diritto a un massimo di 2 giorni di smart working a
settimana, da concordare con il proprio responsabile diretto. Il venerdì è
escluso dal computo se coincide con una chiusura aziendale programmata.
"""

prompt_analisi = f"""Analizza il documento delimitato da triple virgolette e
rispondi in questo formato, un elemento per riga:

<articolo>: <sintesi in massimo 15 parole>

Documento:
'''
{DOCUMENTO}
'''"""

risposta_documento = client.messages.create(
    model="claude-sonnet-5",
    max_tokens=512,
    messages=[{"role": "user", "content": prompt_analisi}]
)

print(risposta_documento.content[0].text)
print(f"\nToken di input consumati (include l'intero documento): {risposta_documento.usage.input_tokens}")


---
## Esercizio — TODO

Modifica il prompt della cella precedente (o scrivine uno nuovo in quella
sottostante) per chiedere al modello una cosa diversa: ad esempio, "elenca
solo le scadenze temporali menzionate nel documento" oppure "riscrivi
l'Articolo 2 in linguaggio più semplice, per un nuovo assunto". Completa i
`# TODO`.

In [ ]:
# TODO: scrivi un nuovo prompt che chieda al modello di estrarre
# SOLO le scadenze temporali (es. "15 giorni di preavviso", "30 giugno") citate nel documento
prompt_scadenze = None  # <-- sostituisci con il tuo prompt

# TODO: invoca client.messages.create() con il nuovo prompt e stampa il risultato


---
## Sfida finale — Assistente con ricerca web integrata

Costruisci una funzione che risponde a una domanda usando lo strumento di
ricerca web lato server di Claude (*grounding*). A differenza degli esempi
precedenti, qui il modello può decidere da solo se e quando cercare
informazioni aggiornate.

Leggi il codice, capiscilo, e completa i frammenti marcati con `# TODO`.

In [ ]:
def chiedi_con_ricerca_web(domanda: str, max_ricerche: int = 3) -> None:
    """
    Pone una domanda a Claude abilitando lo strumento di ricerca web.
    Stampa la risposta finale e, se presenti, le fonti consultate.
    """
    risposta = client.messages.create(
        model="claude-sonnet-5",
        max_tokens=1024,
        tools=[
            {
                "type": "web_search_20260209",
                "name": "web_search",
                # TODO: imposta un limite ragionevole di ricerche per questa domanda
                "max_uses": max_ricerche,
            }
        ],
        messages=[{"role": "user", "content": domanda}]
    )

    for blocco in risposta.content:
        if blocco.type == "text":
            print(blocco.text)
        # TODO: aggiungi un ramo elif per il tipo di blocco "web_search_tool_result"
        # e stampa quante fonti sono state consultate (suggerimento: blocco.content è una lista)


# Prova la funzione con una domanda che richiede informazioni aggiornate
chiedi_con_ricerca_web("Quali sono le principali novità annunciate di recente su Claude di Anthropic?")


### Estensioni opzionali (se finisci prima)

1. **Limita il dominio di ricerca:** usa il parametro `allowed_domains` dello
   strumento `web_search` per restringere la ricerca a un solo sito (es. un
   blog tecnico che conosci).
2. **Confronta con e senza ricerca:** poni la stessa domanda con e senza lo
   strumento `web_search` abilitato, e confronta le due risposte.
3. **Combina ragionamento esteso e ricerca web:** attiva `thinking` insieme
   allo strumento di ricerca sulla stessa richiesta, e osserva come il
   modello pianifica le proprie ricerche prima di rispondere.

In [ ]:
# Spazio per le estensioni opzionali


---
## Autovalutazione della settimana

Rispondi in questa cella prima di chiudere il notebook:

1. **Cosa ha funzionato bene questa settimana?**
   ...

2. **Cosa non hai capito del tutto?**
   ...

3. **Cosa cambieresti nel tuo approccio la settimana prossima?**
   ...

4. **Tempo reale dedicato:**
   ...

---
*Notebook 05 — Modulo 05 del Corso Completo di IA Generativa*